# Módulo 3 · Rebalanceo Óptimo vía Programación Dinámica (Bellman)

Proyecto: Análisis y Diseño de Algoritmos · Optimización de Portafolios

Se resuelve, mediante **backward induction**, la ecuación de Bellman sobre un estado
escalar de exposición al portafolio óptimo (tangente) vs. efectivo, discretizado en una
grilla `Δw`. Esta simplificación mantiene el espacio de estados tratable computacionalmente
frente a un espacio de pesos multidimensional.

In [ ]:
!pip install yfinance plotly -q

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import yfinance as yf
from scipy.optimize import minimize
import json

# ==============================================================
# VARIABLES GLOBALES
# ==============================================================
TICKERS = ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']
START_DATE = '2015-01-01'
END_DATE = '2024-12-31'
CAPITAL = 100000
COSTO_TRANSACCION_PCT = 0.1     # costo lineal (%) por unidad de ajuste
HORIZONTE = 12                  # numero de periodos (meses)
GRID_STEP = 0.01                # resolucion de la grilla de estados
AVERSION_RIESGO = 3.0           # coeficiente de aversion al riesgo (utilidad tipo Merton)

In [ ]:
def cargar_datos(tickers, start, end):
    raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
    if isinstance(raw.columns, pd.MultiIndex):
        precios = raw['Close']
    else:
        precios = raw[['Close']]
        precios.columns = tickers
    precios = precios.dropna(how='all').ffill().dropna()
    tick_list = list(precios.columns)
    log_returns = np.log(precios / precios.shift(1)).dropna()
    mu = log_returns.mean() * 252
    cov = log_returns.cov() * 252
    return precios, log_returns, mu, cov, tick_list

precios, log_returns, mu, cov, TICK_LIST = cargar_datos(TICKERS, START_DATE, END_DATE)

def rendimiento_portafolio(pesos, mu, cov):
    ret = float(np.dot(pesos, mu))
    vol = float(np.sqrt(pesos @ cov @ pesos))
    return ret, vol

def _neg_sharpe(pesos, mu, cov, rf=0.0):
    ret, vol = rendimiento_portafolio(pesos, mu, cov)
    return -(ret - rf) / vol if vol > 0 else 0.0

def optimizar_max_sharpe(mu, cov, rf=0.0):
    n = len(mu)
    bounds = tuple((0.0, 1.0) for _ in range(n))
    cons = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1},)
    init = np.repeat(1.0 / n, n)
    res = minimize(_neg_sharpe, init, args=(mu.values, cov.values, rf),
                    method='SLSQP', bounds=bounds, constraints=cons)
    return res.x if res.success else init

w_tangente = optimizar_max_sharpe(mu, cov)
mu_p, sigma_p = rendimiento_portafolio(w_tangente, mu.values, cov.values)
print(f"Portafolio tangente -> retorno anual: {mu_p*100:.2f}%  |  volatilidad anual: {sigma_p*100:.2f}%")

Portafolio tangente -> retorno anual: 7.74%  |  volatilidad anual: 27.83%


## Formulación del problema de Bellman

Estado $s_t \in [0,1]$: fracción del capital expuesta al portafolio tangente (el resto en
efectivo, retorno 0). Recompensa por periodo tipo utilidad media-varianza (Merton):

$$r(s) = s\,\mu_p - \tfrac{1}{2}\lambda\, s^2 \sigma_p^2$$

Costo de ajuste lineal entre estados consecutivos: $c(s_{t-1}, s_t) = \text{costo}\cdot|s_t - s_{t-1}|$.

La ecuación de Bellman (backward induction) es:

$$J^*_t(s_{t-1}) = \max_{s_t \in \text{grid}} \Big[ r(s_t) - c(s_{t-1}, s_t) + J^*_{t+1}(s_t) \Big], \qquad J^*_T(\cdot) = 0$$

In [ ]:
def ejecutar_dp_rebalanceo(mu_p, sigma_p, costo_trans_pct, horizonte, grid_step, aversion_riesgo=3.0):
    grid = np.round(np.arange(0.0, 1.0 + grid_step, grid_step), 6)
    grid = grid[grid <= 1.0 + 1e-9]
    n_estados = len(grid)

    recompensa = grid * mu_p - 0.5 * aversion_riesgo * (grid ** 2) * (sigma_p ** 2)
    costo_pct = costo_trans_pct / 100.0
    matriz_costos = costo_pct * np.abs(grid.reshape(-1, 1) - grid.reshape(1, -1))

    J = np.zeros((horizonte + 1, n_estados))
    politica = np.zeros((horizonte, n_estados), dtype=int)

    for t in range(horizonte - 1, -1, -1):
        for i in range(n_estados):
            valores = recompensa - matriz_costos[i, :] + J[t + 1, :]
            j_best = int(np.argmax(valores))
            J[t, i] = valores[j_best]
            politica[t, i] = j_best

    return grid, J, politica, matriz_costos

grid, J, politica, matriz_costos = ejecutar_dp_rebalanceo(
    mu_p, sigma_p, COSTO_TRANSACCION_PCT, HORIZONTE, GRID_STEP, AVERSION_RIESGO
)
print(f"Estados discretizados: {len(grid)}  |  Horizonte: {HORIZONTE} periodos")

Estados discretizados: 101  |  Horizonte: 12 periodos


## Resolución y política óptima

Se extrae la trayectoria de decisiones óptimas (`No Cambiar` / `Rebalancear Compra` /
`Rebalancear Venta`) partiendo de un estado inicial neutro ($s_0 = 0.5$).

In [ ]:
estado_actual = int(np.argmin(np.abs(grid - 0.5)))
decisiones = []
for t in range(HORIZONTE):
    siguiente = politica[t, estado_actual]
    delta = grid[siguiente] - grid[estado_actual]
    if abs(delta) < 1e-9:
        decisiones.append('No Cambiar')
    elif delta > 0:
        decisiones.append('Rebalancear Compra')
    else:
        decisiones.append('Rebalancear Venta')
    estado_actual = siguiente

df_acciones = pd.DataFrame({'Periodo': [f'T_{i}' for i in range(HORIZONTE)], 'Decision Optima': decisiones})
df_acciones

,Periodo,Decision Optima
0,T_0,Rebalancear Venta
1,T_1,No Cambiar
2,T_2,No Cambiar
3,T_3,No Cambiar
4,T_4,No Cambiar
5,T_5,No Cambiar
6,T_6,No Cambiar
7,T_7,No Cambiar
8,T_8,No Cambiar
9,T_9,No Cambiar


## Heatmap de costos y simulación de riqueza

Se visualiza la matriz de costos entre todos los pares de estados y se simula hacia
adelante la trayectoria de riqueza de la estrategia DP frente a Buy & Hold y a un
rebalanceo permanente a exposición total ($s=1$).

In [ ]:
grid_labels = [f'{w:.2f}' for w in grid]
fig_heat = px.imshow(matriz_costos,
                      labels=dict(x='Estado Destino (w_t)', y='Estado Origen (w_t-1)', color='Costo Ajuste'),
                      x=grid_labels, y=grid_labels, title='Matriz de Costos de Transicion')
fig_heat.show()

port_log_ret = pd.Series(log_returns.values @ w_tangente, index=log_returns.index)
monthly_log = port_log_ret.resample('M').sum()
retornos_periodicos = np.exp(monthly_log) - 1

def simular_dp_forward(grid, politica, retornos_periodicos, costo_trans_pct, capital, horizonte):
    horizonte = min(horizonte, len(retornos_periodicos))
    retornos = retornos_periodicos.values[:horizonte]
    fechas = retornos_periodicos.index[:horizonte]
    costo_pct = costo_trans_pct / 100.0
    idx_ini = int(np.argmin(np.abs(grid - 0.5)))

    wealth_dp, w_cur, idx_cur = [capital], grid[idx_ini], idx_ini
    for t in range(horizonte):
        r = retornos[t]
        idx_next = politica[t, idx_cur]
        w_next = grid[idx_next]
        costo = costo_pct * abs(w_next - w_cur)
        wealth_dp.append(wealth_dp[-1] * (1 + w_cur * r) * (1 - costo))
        w_cur, idx_cur = w_next, idx_next

    w_fijo = grid[idx_ini]
    wealth_bh = [capital]
    for t in range(horizonte):
        wealth_bh.append(wealth_bh[-1] * (1 + w_fijo * retornos[t]))

    wealth_full = [capital]
    for t in range(horizonte):
        wealth_full.append(wealth_full[-1] * (1 + 1.0 * retornos[t]))

    return fechas, wealth_dp[1:], wealth_bh[1:], wealth_full[1:]

fechas_dp, wealth_dp, wealth_bh_dp, wealth_full_dp = simular_dp_forward(
    grid, politica, retornos_periodicos, COSTO_TRANSACCION_PCT, CAPITAL, HORIZONTE
)

fig_dp = go.Figure()
fig_dp.add_trace(go.Scatter(x=list(range(len(wealth_dp))), y=wealth_dp,
                             name='Estrategia DP (Optima)', line=dict(color='#7C9473', width=3)))
fig_dp.add_trace(go.Scatter(x=list(range(len(wealth_bh_dp))), y=wealth_bh_dp,
                             name='Buy & Hold', line=dict(color='#B3452F', dash='dash')))
fig_dp.add_trace(go.Scatter(x=list(range(len(wealth_full_dp))), y=wealth_full_dp,
                             name='Siempre Rebalanceado (w=1)', line=dict(color='#3D4F4A', dash='dot')))
fig_dp.update_layout(title='Evolucion de Riqueza - Programacion Dinamica',
                      xaxis_title='Periodo', yaxis_title='Valor del Portafolio (USD)')
fig_dp.show()

/tmp/ipykernel_958/4086061834.py:8: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



## Persistencia de resultados

In [ ]:
resultados_m3 = {
    'grid': grid.tolist(),
    'mu_p': mu_p, 'sigma_p': sigma_p,
    'costo_transaccion_pct': COSTO_TRANSACCION_PCT, 'horizonte': HORIZONTE, 'grid_step': GRID_STEP,
    'wealth_dp': {'periodos': list(range(len(wealth_dp))), 'valores': wealth_dp},
    'wealth_bh_dp': {'periodos': list(range(len(wealth_bh_dp))), 'valores': wealth_bh_dp},
    'wealth_full_dp': {'periodos': list(range(len(wealth_full_dp))), 'valores': wealth_full_dp},
    'capital_inicial': CAPITAL,
}

with open('resultados_m3.json', 'w') as f:
    json.dump(resultados_m3, f, indent=2)

print("Resultados del Modulo 3 guardados en resultados_m3.json")

Resultados del Modulo 3 guardados en resultados_m3.json
